In [ ]:
# Carga rápida y selección de columnas

import re
import pandas as pd
import numpy as np

# 1. Carga rápida (solo columnas de auditoría)
RUTA_LOCAL = r"C:\Users\felip\OneDrive\Felipe\Master\TFM\Scripts\fnspid_bancos_v2.parquet"
COLS = [
    "Date", "Stock_symbol", "Raw_Stock_symbol", 
    "Article_title", "Url", "Publisher", 
    "Metodo_Captura", "Detalle_Match", "Fuente_Archivo"
]

df = pd.read_parquet(RUTA_LOCAL, columns=COLS)
print(f"Total filas brutas cargadas: {len(df):,}")
display(df.head())

In [ ]:
# Análisis de fechas y rescate por URL 
# Busca los nulos, extrae fechas ocultas en los enlaces y recorta el dataset estrictamente a la ventana 2010–2023.

# 1. Análisis inicial de fechas
df["Date_clean"] = df["Date"].replace({"nan": np.nan, "None": np.nan, "": np.nan})
df["dt_parsed"] = pd.to_datetime(df["Date_clean"], utc=True, errors="coerce")

nulos_iniciales = df["dt_parsed"].isna().sum()
print(f"Nulos iniciales en fecha: {nulos_iniciales:,}")

# 2. Rescate por URL: En caso de nulos, intenta extraer la fecha de la URL.
mask_na = df["dt_parsed"].isna()
if mask_na.sum() > 0:
    fechas_url = (
        df.loc[mask_na, "Url"]
        .str.extract(r"((?:200\d|201\d|202[0-4])[-/]\d{2}[-/]\d{2})")[0]
        .str.replace("/", "-", regex=False)
    )
    df.loc[mask_na, "dt_parsed"] = pd.to_datetime(fechas_url, utc=True, errors="coerce")

nulos_finales = df["dt_parsed"].isna().sum()
print(f"Fechas rescatadas de la URL: {nulos_iniciales - nulos_finales:,}")

# 3. Filtrar ventana del TFM y descartar datos fuera de rango
df_valid = df.dropna(subset=["dt_parsed"]).copy()
df_valid["year"] = df_valid["dt_parsed"].dt.year
df_2010_2023 = df_valid[(df_valid["year"] >= 2010) & (df_valid["year"] <= 2023)].copy()

print(f"Total registros en ventana 2010-2023: {len(df_2010_2023):,}")

In [ ]:
# Aislamos los registros que quedaron estrictamente fuera de 2010-2023

df_fuera_rango = df_valid[(df_valid["year"] < 2010) | (df_valid["year"] > 2023)]

# Contamos la frecuencia de publicación agrupada por año
distribucion_anios = df_fuera_rango["year"].value_counts().sort_index()
total_descartados = distribucion_anios.sum()

print("=== DISTRIBUCIÓN DE AÑOS DESCARTADOS ===")
print(distribucion_anios)
print("-" * 40)
print(f"Total registros descartados: {total_descartados:,}")

In [ ]:
# Deduplicación estricta. Elimina duplicados dando prioridad a los etiquetados oficiales (EXACT_TICKER) frente a nuestras capturas.

df_2010_2023["date_day"] = df_2010_2023["dt_parsed"].dt.date
df_2010_2023["title_norm"] = df_2010_2023["Article_title"].str.strip().str.lower()

# Sistema de prioridades
prioridad = {
    "EXACT_TICKER": 1, 
    "DIRTY_TICKER": 2, 
    "RESCUED_NO_TICKER": 3, 
    "CROSS_MENTION_OTHER_TICKER": 4
}
df_2010_2023["prio"] = df_2010_2023["Metodo_Captura"].map(prioridad)

antes_dedup = len(df_2010_2023)
df_2010_2023 = (
    df_2010_2023.sort_values("prio")
    .drop_duplicates(subset=["Stock_symbol", "date_day", "title_norm"], keep="first")
    .drop(columns=["prio", "title_norm"])
)

print(f"Duplicados eliminados: {antes_dedup - len(df_2010_2023):,}")
print(f"Noticias únicas limpias: {len(df_2010_2023):,}")

In [ ]:
# 1. FILTRO ANTI-RUIDO: Eliminamos noticias donde el banco actúa como analista

PATRON_RUIDO = re.compile(
    r"\b(?:upgrades?|downgrades?|initiates?\s+coverage|price\s+target|pt\s+to|"
    r"maintains?\s+(?:buy|hold|neutral|sell|overweight|underweight)|"
    r"reiterates?\s+(?:buy|outperform)|top\s+picks?|stocks?\s+to\s+buy|"
    r"strategists?\s+say|analysts?\s+at|favors?\s+these)\b",
    re.IGNORECASE
)
es_ruido_analista = df_2010_2023["Article_title"].fillna("").str.contains(PATRON_RUIDO, regex=True)
df_sin_analistas = df_2010_2023[~es_ruido_analista].copy()

# 2. WHITELIST ESTRICTA: Separamos los datos oficiales de los rescatados
mask_oficial = df_sin_analistas["Metodo_Captura"].isin(["EXACT_TICKER", "DIRTY_TICKER"])
df_oficial = df_sin_analistas[mask_oficial]
df_rescatado = df_sin_analistas[df_sin_analistas["Metodo_Captura"] == "RESCUED_NO_TICKER"].copy()

# Exigimos que las noticias rescatadas contengan el nombre completo de la entidad
nombres_completos = {
    "JPM": r"\b(jpmorgan|jp morgan)\b",
    "BAC": r"\b(bank of america|bofa)\b",
    "C":   r"\b(citigroup|citibank)\b",
    "GS":  r"\b(goldman sachs)\b",
    "MS":  r"\b(morgan stanley)\b",
    "WFC": r"\b(wells fargo)\b"
}

texto_combinado = (df_rescatado["Article_title"].fillna("") + " " + df_rescatado["Url"].fillna("")).str.lower()
mascara_rescate_valido = pd.Series(False, index=df_rescatado.index)

for ticker, patron in nombres_completos.items():
    es_el_ticker = df_rescatado["Stock_symbol"] == ticker
    tiene_nombre_largo = texto_combinado.str.contains(patron, regex=True)
    mascara_rescate_valido = mascara_rescate_valido | (es_el_ticker & tiene_nombre_largo)

df_rescatado_limpio = df_rescatado[mascara_rescate_valido]

# 3. ENSAMBLAJE FINAL
df_limpio = pd.concat([df_oficial, df_rescatado_limpio])

print(f"1. Noticias de analistas eliminadas: {es_ruido_analista.sum():,}")
print(f"2. Basura eliminada por acrónimos ambiguos (MS, C, etc.): {len(df_rescatado) - len(df_rescatado_limpio):,}")
print("-" * 50)
print(f"DATASET FINAL ESTRICTO Y ÚTIL: {len(df_limpio):,}")


In [ ]:
# Matrices de Cobertura y Diagnóstico Final Muestra la densidad de datos rescatados por año y banco para confirmar si hemos tapado los huecos.

print("=" * 80)
print("NUEVA COBERTURA 2010-2023 (EXACT + RESCUED LIMPIO SIN RUIDO DE ANALISTA)")
print("=" * 80)

pivot = pd.pivot_table(
    df_limpio, index="year", columns="Stock_symbol", values="Article_title", aggfunc="count", fill_value=0
).reindex(range(2010, 2024), fill_value=0)
display(pivot)

print("\n" + "=" * 80)
print("DIAGNÓSTICO DE AÑOS FALTANTES TRAS EL RESCATE")
print("=" * 80)

resumen = []
for banco in ["BAC", "C", "GS", "JPM", "MS", "WFC"]:
    s = pivot[banco] if banco in pivot.columns else pd.Series(0, index=range(2010, 2024))
    ceros = s[s == 0].index.tolist()
    resumen.append({
        "Banco": banco,
        "Total Noticias Limpias": int(s.sum()),
        "Años Cubiertos": int((s > 0).sum()),
        "Años con 0 Noticias": ceros if ceros else "Ninguno"
    })

display(pd.DataFrame(resumen))

In [ ]:
# Aislar solo las noticias rescatadas que superaron el filtro anti-ruido
df_rescatadas = df_limpio[df_limpio["Metodo_Captura"] == "RESCUED_NO_TICKER"]

print(f"Total de noticias rescatadas limpias: {len(df_rescatadas):,}")

# Seleccionar columnas clave para que la tabla se lea bien en pantalla
cols_visualizar = ["date_day", "Stock_symbol", "Article_title", "Url"]

# Configurar pandas para que no corte el texto de los titulares largos
pd.set_option('display.max_colwidth', None)

# Mostrar una muestra aleatoria de 10 noticias
display(df_rescatadas[cols_visualizar].sample(10, random_state=42))

In [ ]:
# 1. Patrón específico para detectar "Multiple Sclerosis" camuflado como Morgan Stanley
PATRON_MEDICO_MS = re.compile(
    r"\b(?:multiple\s+sclerosis|pill|drug|treatment|patients?|disease|therapy|biogen|fda|pharma|clinical\s+trials?|dose)\b",
    re.IGNORECASE
)

# 2. Identificar filas donde el banco es MS pero el titular habla de medicina
falsos_positivos_ms = (
    (df_limpio["Stock_symbol"] == "MS") & 
    df_limpio["Article_title"].fillna("").str.contains(PATRON_MEDICO_MS, regex=True)
)

print(f"Falsos positivos de Morgan Stanley (Esclerosis Múltiple) detectados y eliminados: {falsos_positivos_ms.sum():,}")

# 3. Aplicar el filtro para quedarnos solo con lo que NO sea un falso positivo médico
df_limpio = df_limpio[~falsos_positivos_ms]

print(f"NUEVO VOLUMEN DEL DATASET ÚTIL: {len(df_limpio):,}")

In [ ]:
# Problemas principales detectados en el dataset final: confusión de acrónimos (MS, C), ruido de analistas, y fechas faltantes. A pesar de los filtros, 
# aún pueden existir casos residuales de confusión de acrónimos o ruido de analistas. Se recomienda una revisión manual adicional para asegurar la calidad del dataset final.